# 04 · Performance Tables

This notebook loads the pre-computed metric CSVs from `results/tables/` and produces:
- Summary tables (median ± IQR) for NSE, KGE, CRPS, α, and π_rel
- Per-catchment comparison tables (TFT ungauged vs HBV; TFT specialized vs TFT ungauged)
- Empirical non-exceedance probability distribution plots (Fig. 5 and 11 in the paper)

**Prerequisites:** run notebooks `01`, `02`, and `03` first, or use the pre-computed files already in `results/tables/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

RESULTS_DIR = Path("../results/tables")
FIG_DIR = Path("../results/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

METRICS = ["NSE", "KGE", "CRPS", "alpha", "pi_rel"]
METRIC_LABELS = {
    "NSE": "NSE",
    "KGE": "KGE",
    "CRPS": r"CRPS ($\times 10^3$ m$^3$ s$^{-1}$ km$^{-2}$)",
    "alpha": r"Reliability ($\alpha$)",
    "pi_rel": r"Relative resolution ($\pi_{rel}$)",
}

## 1 · Load results

In [ ]:
ungauged = pd.read_csv(RESULTS_DIR / "ungauged_metrics.csv")
specialized = pd.read_csv(RESULTS_DIR / "specialized_metrics.csv")

print(f"Ungauged: {len(ungauged)} rows, models: {ungauged['model'].unique()}")
print(f"Specialized: {len(specialized)} rows, models: {specialized['model'].unique()}")

## 2 · Summary statistics table

In [ ]:
def summary_table(df: pd.DataFrame, group_col: str = "model") -> pd.DataFrame:
    """Compute median and IQR for each metric, grouped by model."""
    rows = []
    for model, grp in df.groupby(group_col):
        row = {"model": model}
        for m in METRICS:
            if m not in grp.columns:
                continue
            vals = grp[m].dropna()
            row[f"{m}_median"] = vals.median()
            row[f"{m}_q25"] = vals.quantile(0.25)
            row[f"{m}_q75"] = vals.quantile(0.75)
        rows.append(row)
    return pd.DataFrame(rows).set_index("model")

summary = summary_table(pd.concat([ungauged, specialized]))
display(summary.round(3))

## 3 · Empirical non-exceedance probability distributions (Fig. 5 / Fig. 11)

In [ ]:
def plot_nep(
    dfs: dict,
    metrics: list,
    title: str,
    filename: str,
) -> None:
    """
    Plot empirical non-exceedance probability distributions for selected metrics.

    Parameters
    ----------
    dfs : dict
        Mapping label -> DataFrame with metric columns.
    metrics : list
        Metric names to plot.
    title : str
        Figure suptitle.
    filename : str
        Output filename (saved to FIG_DIR).
    """
    n = len(metrics)
    fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 4.5), sharey=True)
    linestyles = ["-", "--", "-.", ":"]
    colors = ["#E87722", "#2B7FD4", "#27A844", "#9B59B6"]

    for ax, metric in zip(axes, metrics):
        for (label, df), ls, color in zip(dfs.items(), linestyles, colors):
            vals = df[metric].dropna().sort_values().values
            nep = np.arange(1, len(vals) + 1) / len(vals)
            ax.plot(vals, nep, ls=ls, color=color, lw=1.8, label=label)
        ax.set_xlabel(METRIC_LABELS.get(metric, metric), fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))

    axes[0].set_ylabel("Non-exceedance probability", fontsize=10)
    axes[-1].legend(fontsize=9, loc="lower right")
    fig.suptitle(title, fontsize=12, y=1.02)
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=150, bbox_inches="tight")
    plt.show()


# --- Experiment 1: ungauged TFT vs HBV (full time series) ---
hbv_ung = ungauged[ungauged["model"] == "HBV"]
tft_ung = ungauged[ungauged["model"] == "TFT_ungauged"]

plot_nep(
    dfs={"HBV": hbv_ung, "TFT Ungauged": tft_ung},
    metrics=["NSE", "KGE", "CRPS"],
    title="Ungauged prediction — full time series (Fig. 5)",
    filename="fig05_nep_ungauged.png",
)

# --- Experiment 2: ungauged vs specialized vs HBV (20% test portion) ---
hbv_sp  = specialized[specialized["model"] == "HBV"]
tft_ung_sp = specialized[specialized["model"] == "TFT_ungauged"]
tft_sp  = specialized[specialized["model"] == "TFT_specialized"]

plot_nep(
    dfs={"HBV": hbv_sp, "TFT Ungauged": tft_ung_sp, "TFT Specialized": tft_sp},
    metrics=["NSE", "KGE", "CRPS"],
    title="Specialization — 20% test portion (Fig. 11)",
    filename="fig11_nep_specialized.png",
)

## 4 · Per-catchment comparison table

In [ ]:
# Pivot to wide format: one row per station, columns = (metric, model)
pivot = (
    ungauged
    .pivot_table(index="station", columns="model", values=METRICS[:3])
    .round(3)
)
pivot.columns = [f"{m}_{mdl.replace('TFT_', '')}" for m, mdl in pivot.columns]

# Highlight catchments where TFT outperforms HBV
pivot["TFT_wins_NSE"] = pivot["NSE_ungauged"] > pivot["NSE_HBV"]

print(f"TFT ungauged outperforms HBV on {pivot['TFT_wins_NSE'].sum()} / {len(pivot)} catchments (NSE)")
display(pivot.sort_values("NSE_ungauged", ascending=False).head(20))